In [15]:
!pip install -q google-genai

In [38]:
import json, time
from google.colab import userdata
from google import genai
from google.genai import types

GEMINI_API_KEY = userdata.get('GEMINI_API')
gclient = genai.Client(api_key=GEMINI_API_KEY)
JUDGE_MODEL = "gemini-3.1-flash-lite"

In [39]:
FILE_PATH = 'CON_AUGMENTED_150.json'

with open(FILE_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

print(f"검증 대상: {len(dataset)}개")

검증 대상: 150개


In [44]:
JUDGE_PROMPT = """당신은 지방자치단체 계약서 특수조건 조항을 감사(audit)하는 평가자입니다.

**계약 맥락**
검토 대상은 교육청이 발주하는 SW(소프트웨어) 용역 수의계약의 특수조건입니다.
- "지방계약법"과 그 하위법령(시행령·시행규칙)이 적용되는 지방자치단체 계약이며, 국가계약법이 아닙니다.
- 계약 유형은 "용역"입니다 — 공사·물품·제조 등 다른 계약유형 전용 수치·조문을 clause_text 판단에 잘못 끌어오지 마십시오.
  (예: 지연배상금률처럼 계약유형별로 값이 다른 항목은 refs 안에서도 "용역" 항목에 해당하는 수치만 기준으로 삼아야 합니다.)
- 수의계약이므로 경쟁입찰 관련 조항(입찰공고, 낙찰자 결정방식 등)은 이 감사 범위 밖입니다.

**역할 구분**
- clause_text = 감사 대상. 이 조항이 계약상대자(을)에게 불리하거나 유리하게 법령 기준을 벗어난 독소조항인지 찾아내야 합니다.
- refs = 감사 기준(잣대). refs에 실제로 적힌 조문 원문만이 "정답 기준"입니다.

**중요한 제약**
- 당신이 사전에 알고 있는 지방계약법의 실제 수치·기준(예: 실제 지연배상금률, 실제 하자보수비율 등)은
  절대 사용하지 마십시오. refs에 없는 정보는 "모른다"고 취급하십시오.
- 이 데이터셋에는 RAG 검색이 엉뚱한 조문을 가져온 상황(검토 케이스)이 의도적으로 섞여 있습니다.
  refs가 clause_text와 같은 카테고리를 다루는 것처럼 보여도, 실제로 대조할 수치·요건이 refs 안에
  없다면 당신의 사전 지식으로 그 빈틈을 채우지 말고 반드시 "검토"로 판단하십시오.

**판단 절차**
1. clause_text에서 핵심 수치·조건(요율, 기간, 한도, 승인 방식 등)을 뽑아냅니다.
2. refs 원문 안에서, "용역" 계약유형에 해당하는 대응 기준 수치·조건이 실제로 명시되어 있는지 확인합니다.
   (refs 안에 공사·물품 등 다른 계약유형 수치만 있고 용역 수치가 없다면, 그것도 "대응 기준 없음"으로 취급합니다.)
3. 대응 기준을 찾았다면, clause_text의 수치·조건이 그 기준과 같은지/벗어났는지 비교합니다.
4. 대응 기준을 refs 안에서 찾지 못했다면 곧바로 "검토"로 판단하고 그 이상 진행하지 않습니다.

[판정 기준]
- 일치: clause_text의 핵심 수치·조건이 refs에 명시된 "용역" 기준과 동일하거나 그 범위 안에 있음.
  refs에 없는 부가조건(기간·예외 등)이 clause_text에 추가로 있어도, 핵심 수치가 일치하면 "일치".
- 불일치: clause_text의 핵심 수치·조건이 refs에 명시된 기준에서 벗어남 (계약상대자에게 불리하든 유리하든 모두 불일치 — 방향은 판정값에 반영하지 않고 reasoning에만 서술)
- 검토: refs 안에 clause_text의 핵심 수치·조건과 대조할 "용역" 기준 자체가 없음 (refs가 다른 계약유형·다른 세부사항을 다루거나 일부만 다룸)

반드시 아래 JSON 형식으로만 답하십시오. refs 밖 지식을 근거로 쓰지 마십시오.
{
  "verdict": "일치" 또는 "불일치" 또는 "검토",
  "reasoning": "clause_text의 어느 수치·조건을, refs의 어느 항/호와 대조했는지(또는 refs에 대조 기준이 없다는 점을) 한두 문장으로"
}
"""

def judge_item(item, retries=3):
    payload = {
        "clause_text": item.get("clause_text", ""),
        "refs": item.get("refs", []),
    }
    user_msg = json.dumps(payload, ensure_ascii=False)

    for attempt in range(retries):
        try:
            resp = gclient.models.generate_content(
                model=JUDGE_MODEL,
                contents=f"{JUDGE_PROMPT}\n\n판정 대상:\n{user_msg}",
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            print("RAW RESPONSE:", resp.text[:300])  # 디버깅용 — 확인 후 지워도 됨
            return json.loads(resp.text)
        except Exception as e:
            print(f"  [warn] judge 호출 실패(attempt {attempt+1}): {e}")
            time.sleep(5)
    return {"verdict": "ERROR", "reasoning": "judge 호출 실패"}

In [45]:
results = []
mismatches = []

for i, item in enumerate(dataset):
    original_verdict = item['label']
    gemini = judge_item(item)
    predicted_verdict = gemini.get('verdict')

    agree = (predicted_verdict == original_verdict)
    results.append({
        "id": item.get("id"),
        "gpt_label": original_verdict,
        "gemini_label": predicted_verdict,
        "agree": agree,
        "reasoning": gemini.get("reasoning"),
    })

    tag = "일치" if agree else "불일치"
    print(f"[{i+1}/{len(dataset)}] {item.get('id')} | GPT={original_verdict} vs Gemini={predicted_verdict} -> {tag}")

    if not agree:
        mismatches.append(item.get("id"))

    time.sleep(7)

    with open('agreement_results_con.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

agreement_rate = sum(r['agree'] for r in results) / len(results)
print(f"\n총 {len(dataset)}개 중 일치 {sum(r['agree'] for r in results)}개 ({agreement_rate:.1%})")
print("불일치 id 목록:", mismatches)

RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 지연배상금률(1000분의 1.3)은 refs의 지방계약법 시행규칙 제75조 제3호(용역: 1000분의 1.3)와 일치하며, 지체상금 한도(100분의 30)는 지방계약법 시행령 제90조 제3항의 기준과 일치합니다."
}
[1/150] 지체상금_일치_001 | GPT=일치 vs Gemini=일치 -> 일치
RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 지연배상금률 1000분의 1.3은 지방계약법 시행규칙 제75조 제3호에서 규정하는 '용역' 계약의 지연배상금률 기준과 일치합니다."
}
[2/150] 지체상금_일치경계_001 | GPT=일치 vs Gemini=일치 -> 일치
RAW RESPONSE: {
  "verdict": "불일치",
  "reasoning": "clause_text는 지연배상금률을 1000분의 3으로 규정하고 있으나, 지방계약법 시행규칙 제75조 제3호에 따른 용역 계약의 기준인 1000분의 1.3과 일치하지 않습니다."
}
[3/150] 지체상금_불일치_을불리_A_001 | GPT=불일치 vs Gemini=불일치 -> 일치
RAW RESPONSE: {
  "verdict": "검토",
  "reasoning": "clause_text는 지연배상금률(1000분의 2)을 규정하고 있으나, 제공된 refs(지방계약법 시행령 제64조, 제67조)에는 용역 계약의 지연배상금률에 관한 기준이 명시되어 있지 않아 대조가 불가능합니다."
}
[4/150] 지체상금_검토_001 | GPT=검토 vs Gemini=검토 -> 일치
RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 대가 지급 기한(5일)은 지방계약법 시행령 제67조 제1항에서 규정하는 대가 지급 기한(5일)과 일치합니다."
}
[5/150] 대금지급_일치

In [47]:
results = []
mismatches = []

for i, item in enumerate(dataset):
    original_verdict = item['label']
    gemini = judge_item(item)
    predicted_verdict = gemini.get('verdict')

    agree = (predicted_verdict == original_verdict)
    results.append({
        "id": item.get("id"),
        "gpt_label": original_verdict,
        "gemini_label": predicted_verdict,
        "agree": agree,
        "reasoning": gemini.get("reasoning"),
    })

    tag = "일치" if agree else "불일치"
    print(f"[{i+1}/{len(dataset)}] {item.get('id')} | GPT={original_verdict} vs Gemini={predicted_verdict} -> {tag}")

    if not agree:
        mismatches.append(item.get("id"))

    time.sleep(7)

    with open('agreement_results_con.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

agreement_rate = sum(r['agree'] for r in results) / len(results)
print(f"\n총 {len(dataset)}개 중 일치 {sum(r['agree'] for r in results)}개 ({agreement_rate:.1%})")
print("불일치 id 목록:", mismatches)

RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 지연배상금률(1000분의 1.3)은 refs의 지방계약법 시행규칙 제75조 제3호(용역: 1000분의 1.3)와 일치하며, 지체상금 한도(100분의 30)는 지방계약법 시행령 제90조 제3항의 기준과 일치합니다."
}
[1/150] 지체상금_일치_001 | GPT=일치 vs Gemini=일치 -> 일치
RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 지연배상금률 1000분의 1.3은 지방계약법 시행규칙 제75조 제3호에서 규정하는 '용역' 계약의 지연배상금률 기준과 일치합니다."
}
[2/150] 지체상금_일치경계_001 | GPT=일치 vs Gemini=일치 -> 일치
RAW RESPONSE: {
  "verdict": "불일치",
  "reasoning": "clause_text는 지연배상금률을 1000분의 3으로 규정하고 있으나, 지방계약법 시행규칙 제75조 제3호에 따른 용역 계약의 기준인 1000분의 1.3과 일치하지 않습니다."
}
[3/150] 지체상금_불일치_을불리_A_001 | GPT=불일치 vs Gemini=불일치 -> 일치
RAW RESPONSE: {
  "verdict": "검토",
  "reasoning": "clause_text는 지연배상금률(1000분의 2)을 규정하고 있으나, 제공된 refs(지방계약법 시행령 제64조, 제67조)에는 용역 계약의 지연배상금률에 관한 기준이 명시되어 있지 않아 대조가 불가능합니다."
}
[4/150] 지체상금_검토_001 | GPT=검토 vs Gemini=검토 -> 일치
RAW RESPONSE: {
  "verdict": "일치",
  "reasoning": "clause_text의 대가 지급 기한(5일)은 지방계약법 시행령 제67조 제1항에서 규정하는 대가 지급 기한(5일)과 일치합니다."
}
[5/150] 대금지급_일치

In [48]:
from collections import Counter
confusion = Counter((r['gpt_판정'], r['gemini_판정']) for r in results)
print("혼동행렬 (GPT라벨, Gemini라벨) -> 개수")
for k, v in sorted(confusion.items(), key=lambda x: -x[1]):
    marker = "  <- 불일치" if k[0] != k[1] else ""
    print(f"  {k}: {v}{marker}")

KeyError: 'gpt_판정'